### 1. Basic Tasks

In [0]:
-- 1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table.

In [0]:
create table if not exists cyntexa_dev.bronze.customers (customer_id int, name string, email string, city string, state string, updated_at timestamp default current_timestamp())
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')

In [0]:
insert into cyntexa_dev.bronze.customers (customer_id, name, email, city, state) values 
(101, 'Aarav Sharma', 'aarav@gmail.com', 'Jaipur', 'Rajasthan'),
(102, 'Ananya Gupta', 'ananya@gmail.com', 'Delhi', 'Delhi'),
(103, 'Rohan Mehta', 'rohan@gmail.com', 'Mumbai', 'Maharashtra'),
(104, 'Priya Singh', 'priya@gmail.com', 'Pune', 'Maharashtra'),
(105, 'Aditya Verma', 'aditya@gmail.com', 'Bangalore', 'Karnataka'),
(106, 'Neha Jain', 'neha@gmail.com', 'Ahmedabad', 'Gujarat'),
(107, 'Raj Malhotra', 'raj@gmail.com', 'Chandigarh', 'Chandigarh'),
(108, 'Isha Patel', 'isha@gmail.com', 'Surat', 'Gujarat'),
(109, 'Kunal Agarwal', 'kunal@gmail.com', 'Kolkata', 'West Bengal'),
(110, 'Simran Kaur', 'simran@gmail.com', 'Ludhiana', 'Punjab');

In [0]:
create table cyntexa_dev.bronze.customers_scd as select * except (updated_at) from cyntexa_dev.bronze.customers

In [0]:
merge into cyntexa_dev.bronze.customers_scd t 
using (
    select * from cyntexa_dev.bronze.customers 
    qualify rank() over (partition by customer_id order by updated_at desc) = 1
) s
on t.customer_id = s.customer_id
when matched then 
update set t.name = s.name,t.email = s.email , t.city = s.city, t.state = s.state
when not matched then 
insert(customer_id, name, email, city, state) values (s.customer_id, s.name, s.email, s.city, s.state)

In [0]:
select * from cyntexa_dev.bronze.customers_scd where customer_id = 101

In [0]:
insert into cyntexa_dev.bronze.customers (customer_id, name, email, city, state) values 
(101, 'Aarav Sharma', 'aarav@gmail.com', 'Mumbai',"Maharashtra")

Now run MERGE INTO script again to update in scd table

In [0]:
select * from cyntexa_dev.bronze.customers_scd where customer_id = 101

Now the customers_scd table contains updated value for customer_id 101

In [0]:
-- 2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity Catalog permissions.

Created 2 groups Data Engineers and Data Analyst in Identity and Access management now grantig following permisions
- Data Engineers group: Can see all tables/ columns wihtout masking
- Data Analyst group: can see only silver/masked tables aor views 

In [0]:
grant select 
on table cyntexa_dev.bronze.customers
to `data_engineers` 

In [0]:
create function cyntexa_dev.bronze.mask_email(email string)
returns string
return concat(
    substring(email,1,2),
    '****',
    substring(email,instr(email,'@'),length(email))
)

In [0]:
create view masked_customer as
select customer_id, name, cyntexa_dev.bronze.mask_email(email) as email, city,
state from cyntexa_dev.bronze.customers

In [0]:
grant select 
on view masked_customer
to `data_analysts`

In [0]:
-- 3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms, what a DBU is billing for.

DBU stands for Databricks Units and is a unit used to measure Databricks compute consumption. The number of DBUs consumed depends on the type and configuration of the compute resource and how long it runs.

DBUs measure the amount of Databricks compute capaciry used and calculates compute cost.

### 2. Intermediate Tasks

In [0]:
-- 4. Build a full SCD Type 2 table: implement the MERGE that closes out old records (setting end_date and is_current) and inserts new versions when a tracked column changes.

In [0]:
create table cyntexa_dev.silver.products (
    product_id int, 
    name string,
    category string,
    price double
)

In [0]:
INSERT INTO cyntexa_dev.silver.products
VALUES
(1, 'Product1', 'Category1', 100.00),
(2, 'Product2', 'Category2', 150.00),
(3, 'Product3', 'Category1', 200.00),
(4, 'Product4', 'Category2', 250.00),
(5, 'Product5', 'Category1', 300.00),
(6, 'Product6', 'Category2', 350.00),
(7, 'Product7', 'Category1', 400.00),
(8, 'Product8', 'Category2', 450.00),
(9, 'Product9', 'Category1', 500.00),
(10, 'Product10', 'Category2', 550.00);

In [0]:
create table cyntexa_dev.silver.products_scd2 as 
select *,
cast(null as date) as effective_date,
cast(null as date) as end_date,
cast(null as int) as version,
cast(null  as boolean) as is_current
from cyntexa_dev.silver.products
where 1 = 0

In [0]:
merge into cyntexa_dev.silver.products_scd2 t
using cyntexa_dev.silver.products s 
on t.product_id = s.product_id and t.is_current = true
when matched and (not(t.name <=> s.name) or
not(t.category <=> s.category) or
not(t.price <=> s.price))
then update set
t.end_date = current_date(),
t.is_current = false
when not matched then
insert (product_id, name, category, price, effective_date, end_date, version, is_current)
values (s.product_id, s.name, s.category, s.price, current_date(), null, 1, true);

insert into cyntexa_dev.silver.products_scd2
select
s.product_id,
s.name,
s.category,
s.price,
current_date() as effective_date,
null as end_date,
coalesce(max(t.version),0) + 1 as version,
true as is_current
from cyntexa_dev.silver.products s
inner join cyntexa_dev.silver.products_scd2 t 
on t.product_id = s.product_id and t.end_date = current_date()
where not(t.name <=> s.name) or
not(t.category <=> s.category) or
not(t.price <=> s.price)
group by s.product_id,
s.name,
s.category,
s.price

In [0]:
select * from cyntexa_dev.silver.products_scd2

In [0]:
insert into cyntexa_dev.silver.products values(1, 'Product1', 'Category5', 110.00)

In [0]:
-- 5. Query the SCD Type 2 table to answer a point-in-time question, e.g. 'what was this customer's address as of March 1st?'
select * from cyntexa_dev.silver.products_scd2
where product_id = 1
and effective_date <= '2026-08-30' and end_date is null

In [0]:
-- 6. Compare the estimated DBU cost of running a job on all-purpose vs. job compute, and recommend which Cyntexa should use for its nightly pipeline.

All purpose compute vs Job compute
- All purpose compute: designed for interactive work such as developting, testing and running notebooks manually 
- Job compute: Designed specifically automated jobs and scheduled pipelies, where compute is started for the job and terminated with completion

All purpose Computes costs more than Jpb compute for the same workload because it is intended for interactive usage.

For Cyntexa's nightly pipeline, choose Job compute because the pipeline is an automated, scheduled workload and does not require and interactive cluster to remain available. This provide bettere cost effeciency and avoids paying for compute when pipeline is not running

###3. Advanced Tasks

In [0]:
-- 7. Design a governance model for Cyntexa: which columns across which tables are sensitive (PII), which Unity Catalog groups should have access, and how you'd audit access after the fact.

In [0]:
-- 8. Extend the SCD Type 2 pattern to track changes across 3+ columns simultaneously, and handle the edge case of a customer record that hasn't changed since the last load (it should not create a false new version).

In [0]:
select * from cyntexa_dev.silver.products where product_id = 10

In [0]:
-- inserted updated product_id 10 with 3+ updates 
insert into cyntexa_dev.silver.products values (10, "Product_10", "Category3", 600)

In [0]:
select * from cyntexa_dev.silver.products_scd2 where product_id = 10 

The SCD2 implementation handles 3 or more updation into single version without creating multiple versions to update each  column one by one 

In [0]:
-- inserting duplicate row 
insert into cyntexa_dev.silver.products values (10, "Product_10", "Category3", 600)

In [0]:
select * from cyntexa_dev.silver.products_scd2 where product_id = 10